In [1]:
import os
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

transport = httpx.HTTPTransport(verify=False)
http_client = httpx.Client(transport=transport)

llm = ChatOpenAI(
    base_url = BASE_URL,
    api_key = API_KEY,
    model = MODEL,
    http_client = http_client,
    streaming = False,
)

In [10]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate

# class WeatherResponse(BaseModel):
#     conditions: str

checkpointer = InMemorySaver()

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

prompt = ChatPromptTemplate.from_messages([
    ("user", "you are a helpful assistant")
])

agent = create_react_agent(
    model = llm,
    tools = [get_weather],
    checkpointer=checkpointer,
    prompt = prompt,
    # response_format = WeatherResponse
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    config
)

# response["structured_response"]


In [11]:
response

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='9b252f73-266c-4d8e-b43f-76db3db88589'),
  AIMessage(content="I'm here to help! How can I assist you today? 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 155, 'total_tokens': 319, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 148, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '0199815beb604c2119d62a89606c914d', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--7c416f73-24b6-42c6-8974-61fa05c52223-0', usage_metadata={'input_tokens': 155, 'output_tokens': 164, 'total_tokens': 319, 'input_token_details': {}, 'output_token_details': {'reasoning': 148}})]}